In [2]:
import tflearn
import tensorflow as tf
import numpy as np
import sys
import os
import random
from essentia.standard import MonoLoader, Windowing, Spectrum, MelBands
from collections import deque
from zipfile import ZipFile
import csv

from chop_song import process_song

def main():
    '''
    将歌曲切分成23ms片段，并对每个片段进行STFT变换，通过mel尺度滤波器压缩至80个频带，并对数缩放反映人类的响度感知
    '''
    # 最后输出一个同名npy文件
    songfile = "Take Over.mp3"
    song = songfile[: - 4]
    npy_file = song + " Input.npy"

    # 防止一下数据重复处理
    dir_list = os.listdir()

    if dir_name not in dir_list:
        print("Chopping up your song..")
        process_song(songfile)
    else:
        print("Song already chopped, moving on..")

    # 处理歌曲数据
    print("Analyzing your song..")
    analyze_song(npy_file, dir_name)
    
    return

def create_analyzers(fs=44100.0,
                     nhop=1024,
                     nffts=[1024, 2048, 4096],
                     mel_nband=80,
                     mel_freqlo=27.5,
                     mel_freqhi=16000.0):
    '''
    从DDC改编而来
    https://arxiv.org/abs/1703.06891
    '''
    analyzers = []
    for nfft in nffts:
        window = Windowing(size=nfft, type='blackmanharris62')
        spectrum = Spectrum(size=nfft)
        mel = MelBands(inputSize=(nfft // 2) + 1,
                       numberBands=mel_nband,
                       lowFrequencyBound=mel_freqlo,
                       highFrequencyBound=mel_freqhi,
                       sampleRate=fs)
        analyzers.append((window, spectrum, mel))
    return analyzers[0][0], analyzers[0][1], analyzers[0][2]

def analyze_song(file_name = None, dir_name = None):
        '''
        write something here
        '''
        file_list = os.listdir()
        for f in file_list:
            if f == file_name:
                print("Song already has been processed, exiting processing..")
                #os.chdir(cwd)
                return

        cwd = os.getcwd()
        if dir_name != None:
            new_dir = cwd + "/" + dir_name
            os.chdir(new_dir)

        file_list = os.listdir()
        window, spectrum, mel = create_analyzers()
        feats_list = []
        i = 0
        
        for fn in file_list:
            if fn[len(fn) - 1] != 'v':
                continue
            try:
                loader = MonoLoader(filename=fn, sampleRate=44100.0)
                samples = loader()
                feats = window(samples)
                if len(feats) % 2 != 0:
                    feats = np.delete(feats, random.randint(0, len(feats) - 1))
                feats = spectrum(feats)
                feats = mel(feats)
                feats_list.append(feats)
                i+=1
            except Exception as e:
                feats_list.append(np.zeros(80, dtype=np.float32))
                i += 1

        # Apply numerically-stable log-scaling
        feats_list = np.array(feats_list)
        feats_list = np.log(feats_list + 1e-16)
        print(len(feats_list), "length of feats list")
        print(type(feats_list[0][0]))
        if dir_name != None:
            os.chdir(cwd)
        np.save(file_name, feats_list)
        return


ModuleNotFoundError: No module named 'tflearn'

In [1]:
%lsmagic

Available line magics:
%alias  %alias_magic  %autoawait  %autocall  %automagic  %autosave  %bookmark  %cd  %clear  %cls  %colors  %conda  %config  %connect_info  %copy  %ddir  %debug  %dhist  %dirs  %doctest_mode  %echo  %ed  %edit  %env  %gui  %hist  %history  %killbgscripts  %ldir  %less  %load  %load_ext  %loadpy  %logoff  %logon  %logstart  %logstate  %logstop  %ls  %lsmagic  %macro  %magic  %matplotlib  %mkdir  %more  %notebook  %page  %pastebin  %pdb  %pdef  %pdoc  %pfile  %pinfo  %pinfo2  %pip  %popd  %pprint  %precision  %prun  %psearch  %psource  %pushd  %pwd  %pycat  %pylab  %qtconsole  %quickref  %recall  %rehashx  %reload_ext  %ren  %rep  %rerun  %reset  %reset_selective  %rmdir  %run  %save  %sc  %set_env  %store  %sx  %system  %tb  %time  %timeit  %unalias  %unload_ext  %who  %who_ls  %whos  %xdel  %xmode

Available cell magics:
%%!  %%HTML  %%SVG  %%bash  %%capture  %%cmd  %%debug  %%file  %%html  %%javascript  %%js  %%latex  %%markdown  %%perl  %%prun  %%pypy  %%python 